# 02 - Active vs. Total Parameter Cost Demo

Companion notebook to `01-mixtral-architecture-deep-dive.md`. Makes the MoE efficiency argument
numerically instead of just asserting it in prose: given Mixtral's real, published shape (~47B total
parameters, ~13B active parameters per token), this notebook computes the inference FLOPs implication
against an equivalently-sized dense model, and against a dense model sized to match Mixtral's *active*
parameter count.

All figures are either Mixtral's real, published total/active parameter counts, or clearly-labeled
illustrative unit-cost assumptions used only to show the *shape* of the cost argument -- not a verified
production cost model. No network calls, no real model -- pure arithmetic.

## 1. Mixtral's real, published parameter shape

~47B total parameters; ~13B active parameters per token (top-2-of-8 routing, chapter 01). We also
define two comparison points: a dense model with the *same total* parameter count (all of it active on
every token), and a dense model with the *same active* parameter count as Mixtral.

In [1]:
MIXTRAL_TOTAL_PARAMS = 47e9   # Mixtral 8x7B's real, published total parameter count (approx.)
MIXTRAL_ACTIVE_PARAMS = 13e9  # Mixtral 8x7B's real, published active parameters per token (approx.)

DENSE_MATCHED_TOTAL = MIXTRAL_TOTAL_PARAMS    # a dense model with the same total param count (all active)
DENSE_MATCHED_ACTIVE = MIXTRAL_ACTIVE_PARAMS  # a dense model with the same active param count as Mixtral

print("Mixtral 8x7B total parameters:            {:6.1f}B".format(MIXTRAL_TOTAL_PARAMS / 1e9))
print("Mixtral 8x7B active parameters per token:  {:6.1f}B".format(MIXTRAL_ACTIVE_PARAMS / 1e9))
print("Active fraction of total:                  {:5.1f}%".format(MIXTRAL_ACTIVE_PARAMS / MIXTRAL_TOTAL_PARAMS * 100))


Mixtral 8x7B total parameters:              47.0B
Mixtral 8x7B active parameters per token:    13.0B
Active fraction of total:                   27.7%


## 2. FLOPs per token: a standard, well-known approximation

A widely-used, standard approximation for a Transformer's inference FLOPs per token is roughly
`2 * (number of active parameters)` -- each parameter participates in roughly one multiply-add (2 FLOPs)
per token during a forward pass. This is a standard order-of-magnitude approximation used broadly in the
literature (it undercounts some overhead like attention-score computation, and overcounts a bit for
parameters that don't touch every token) -- good enough to compare relative costs, not intended as an
exact hardware-level FLOPs count. We apply it to all three models: Mixtral (using its *active* count),
the total-matched dense model (using its full parameter count, since a dense model activates every
parameter on every token), and the active-matched dense model.

In [2]:
FLOPS_PER_PARAM_PER_TOKEN = 2  # standard order-of-magnitude approximation (one multiply-add)

def inference_flops_per_token(active_params):
    return FLOPS_PER_PARAM_PER_TOKEN * active_params

mixtral_flops = inference_flops_per_token(MIXTRAL_ACTIVE_PARAMS)
dense_matched_total_flops = inference_flops_per_token(DENSE_MATCHED_TOTAL)     # dense model, all 47B active
dense_matched_active_flops = inference_flops_per_token(DENSE_MATCHED_ACTIVE)   # dense model, 13B active

header = "{:45s} {:>15s} {:>15s}".format("Model", "Active params", "FLOPs/token")
print(header)
print("-" * 78)
print("{:45s} {:14.1f}B {:15.3e}".format("Mixtral 8x7B (sparse MoE, top-2 of 8)", MIXTRAL_ACTIVE_PARAMS / 1e9, mixtral_flops))
print("{:45s} {:14.1f}B {:15.3e}".format("Dense model, same TOTAL params (47B, all active)", DENSE_MATCHED_TOTAL / 1e9, dense_matched_total_flops))
print("{:45s} {:14.1f}B {:15.3e}".format("Dense model, same ACTIVE params as Mixtral (13B)", DENSE_MATCHED_ACTIVE / 1e9, dense_matched_active_flops))


Model                                           Active params     FLOPs/token
------------------------------------------------------------------------------
Mixtral 8x7B (sparse MoE, top-2 of 8)                   13.0B       2.600e+10
Dense model, same TOTAL params (47B, all active)           47.0B       9.400e+10
Dense model, same ACTIVE params as Mixtral (13B)           13.0B       2.600e+10


## 3. The efficiency argument, stated numerically

Two comparisons matter, and they make two different points:

- **Mixtral vs. the total-matched dense model (47B, all active):** how much inference compute Mixtral
  *saves* versus a dense model with the same total learned capacity.
- **Mixtral vs. the active-matched dense model (13B active):** confirms Mixtral's inference cost is
  essentially the *same* as a much smaller dense model, while carrying far more total parameters/capacity.

In [3]:
compute_savings_vs_total_matched = 1 - (mixtral_flops / dense_matched_total_flops)
capacity_multiple_vs_active_matched = MIXTRAL_TOTAL_PARAMS / DENSE_MATCHED_ACTIVE
flops_ratio_vs_active_matched = mixtral_flops / dense_matched_active_flops

print("Comparison 1: Mixtral vs. a dense model with the SAME TOTAL parameter count (47B, all active)")
print("  Mixtral's inference FLOPs/token:     {:.3e}".format(mixtral_flops))
print("  Dense (47B, all active) FLOPs/token: {:.3e}".format(dense_matched_total_flops))
print("  --> Mixtral uses {:.1f}% LESS inference compute per token".format(compute_savings_vs_total_matched * 100))
print("      than a dense model with the same total learned capacity.")
print()
print("Comparison 2: Mixtral vs. a dense model with the SAME ACTIVE parameter count (13B)")
print("  FLOPs ratio (Mixtral / dense-13B): {:.3f}  (should be ~1.0 -- same active compute)".format(flops_ratio_vs_active_matched))
print("  But Mixtral carries {:.2f}x the TOTAL learned parameters/capacity".format(capacity_multiple_vs_active_matched))
print("      of that same-active-cost dense model, for essentially the same inference compute.")

assert compute_savings_vs_total_matched > 0.5, "Mixtral should use well under half the compute of the total-matched dense model."
assert abs(flops_ratio_vs_active_matched - 1.0) < 1e-9, "Mixtral's active-parameter FLOPs should equal the active-matched dense model's FLOPs exactly."
print()
print("Both assertions hold -- this is the numeric version of chapter 01's claim: Mixtral gets access")
print("to near-dense-47B-model capacity at close to dense-13B-model inference compute cost.")


Comparison 1: Mixtral vs. a dense model with the SAME TOTAL parameter count (47B, all active)
  Mixtral's inference FLOPs/token:     2.600e+10
  Dense (47B, all active) FLOPs/token: 9.400e+10
  --> Mixtral uses 72.3% LESS inference compute per token
      than a dense model with the same total learned capacity.

Comparison 2: Mixtral vs. a dense model with the SAME ACTIVE parameter count (13B)
  FLOPs ratio (Mixtral / dense-13B): 1.000  (should be ~1.0 -- same active compute)
  But Mixtral carries 3.62x the TOTAL learned parameters/capacity
      of that same-active-cost dense model, for essentially the same inference compute.

Both assertions hold -- this is the numeric version of chapter 01's claim: Mixtral gets access
to near-dense-47B-model capacity at close to dense-13B-model inference compute cost.


## 4. Illustrative cost-per-million-tokens, to make the efficiency tangible

**Clearly labeled as illustrative** -- we assume a made-up, round `$ per PFLOP` unit cost, applied
identically to all three models, purely to translate the FLOPs comparison above into a cost-shaped
number. The absolute dollar figures below are not a claim about real GPU or API pricing -- only the
*relative* comparison between the three rows is meaningful, and that relative comparison is exactly the
one computed with real FLOPs above.

In [4]:
# Illustrative unit cost -- NOT a verified real GPU/API price, chosen only to make the comparison tangible.
ILLUSTRATIVE_COST_PER_PFLOP = 0.02   # made-up $ per 1e15 FLOPs, applied identically to every row
TOKENS_PER_MILLION = 1_000_000

def illustrative_cost_per_million_tokens(flops_per_token):
    total_flops = flops_per_token * TOKENS_PER_MILLION
    pflops = total_flops / 1e15
    return pflops * ILLUSTRATIVE_COST_PER_PFLOP

rows = [
    ("Mixtral 8x7B (sparse MoE)", mixtral_flops),
    ("Dense, same TOTAL params (47B)", dense_matched_total_flops),
    ("Dense, same ACTIVE params (13B)", dense_matched_active_flops),
]

print("Illustrative cost per 1M tokens processed (made-up unit cost, relative comparison only):")
print("{:40s} {:>30s}".format("Model", "$ / 1M tokens (illustrative)"))
print("-" * 72)
for label, flops in rows:
    cost = illustrative_cost_per_million_tokens(flops)
    print("{:40s} {:29.4f}".format(label, cost))

print()
print("This is the numeric shape of chapter 01's efficiency claim: at this platform's document volume")
print("(course 13, chapter 02), that per-token compute gap compounds directly into the cost-at-volume")
print("argument for self-hosting a sparse MoE model instead of an equivalently-capable dense one.")


Illustrative cost per 1M tokens processed (made-up unit cost, relative comparison only):
Model                                      $ / 1M tokens (illustrative)
------------------------------------------------------------------------
Mixtral 8x7B (sparse MoE)                                       0.5200
Dense, same TOTAL params (47B)                                  1.8800
Dense, same ACTIVE params (13B)                                 0.5200

This is the numeric shape of chapter 01's efficiency claim: at this platform's document volume
(course 13, chapter 02), that per-token compute gap compounds directly into the cost-at-volume
argument for self-hosting a sparse MoE model instead of an equivalently-capable dense one.


## Summary

| Section | Computes | Matches |
|---|---|---|
| 1 | Mixtral's real total (~47B) vs. active (~13B) parameter counts | Chapter 01 |
| 2 | Standard FLOPs-per-token approximation applied to Mixtral and two dense comparison points | Chapter 01's "active vs. total" efficiency claim |
| 3 | Numerically confirms Mixtral matches a ~13B dense model's inference cost while carrying ~47B of capacity | The core MoE efficiency argument, stated numerically rather than asserted |
| 4 | Translates the FLOPs comparison into an illustrative relative cost-per-token figure | Course 13, chapter 02's cost-at-volume rationale for self-hosting |

The takeaway, stated precisely: Mixtral's sparse MoE design is not a free lunch that magically avoids
all cost -- it's a specific, quantifiable trade of extra total parameters (memory/storage) for reduced
active-compute-per-token (inference cost), and that trade is exactly what sections 2-3 make concrete.